# Auto-SpyDR PoC: AI Agent для генерации Gherkin тестов

Минимальный baseline для перевода ручных тест-кейсов в Gherkin формат с помощью LLM.

## Workflow
1. Загружаем доступные Gherkin шаги
2. Вводим текст ручного тест-кейса
3. LLM генерирует .feature файл на основе доступных шагов

## 1. Setup

In [7]:
# Установка зависимостей (раскомментировать при первом запуске)
# !pip install openai python-dotenv

import os
from openai import OpenAI
from dotenv import load_dotenv

# Загрузка API ключа из .env файла
load_dotenv()

# Инициализация клиента OpenAI
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ Setup complete")

✓ Setup complete


## 2. Доступные Gherkin шаги

Определяем шаги, которые агент может использовать при генерации тестов.

In [24]:
# Доступные Gherkin шаги (ПОЛНЫЙ СПИСОК, извлеченный автоматически)
AVAILABLE_STEPS = {
    "given": [
        "I buy {amount:Number} apples",
        "I create a symlink from \"{source}\" to \"{dest}\"",
        "I dispatch an async-call with param \"{param}\"",
        "I ensure that the directory \"{directory:Path}\" does not exist",
        "I ensure that the directory \"{directory:Path}\" exists",
        "I inspect the tags of the current scenario",
        "I meet \"{person_name:w} at the {location:w}\"",
        "I meet \"{person_name:w}\"",
        "I need {word} scenario setup",
        "I provide a/an \"{bigdecimal}\" value as bigdecimal",
        "I provide a/an \"{biginteger}\" value as biginteger",
        "I provide a/an \"{byte}\" value as byte",
        "I provide a/an \"{double}\" value as double",
        "I provide a/an \"{float}\" value as float",
        "I provide a/an \"{int}\" value as int",
        "I provide a/an \"{long}\" value as long",
        "I provide a/an \"{short}\" value as short",
        "I provide a/an \"{word}\" value as word",
        "I provide a/an \"{}\" value as any",
        "I provide a/an {string} value as string",
        "I sell {amount:Number} {fruit:Fruit}",
        "I setup the console encoding to \"{encoding:Unquoted}\"",
        "I setup the console encoding to \"{encoding:Unquoted}\" for language \"{language:Unquoted}\"",
        "I setup the current values for active tags with",
        "I setup the current values for active tags with:",
        "I use the current directory as working directory",
        "a behave model with",
        "a behave model with:",
        "a directory named \"{path:Path}\"",
        "a file named \"{filename:Path}\" should exist",
        "a file named \"{filename:Path}\" should not exist",
        "a file named \"{filename:Path}\" using encoding=\"{encoding:Unquoted}\" with",
        "a file named \"{filename:Path}\" using encoding=\"{encoding:Unquoted}\" with:",
        "a file named \"{filename:Path}\" with",
        "a file named \"{filename:Path}\" with:",
        "a file named \"{filename}\" with",
        "a minimum number value of \"{min_value:d}\"",
        "a new working directory",
        "a person named \"{name}\"",
        "a person named {string}",
        "a step with name=\"{name}\"",
        "a step with table data",
        "a step with table data:",
        "a string {argument:custom} a custom type",
        "a string {argument} an argument",
        "an async-step passes",
        "an empty file named \"{filename:Path}\"",
        "behave has the following feature fileset",
        "behave has the following feature fileset:",
        "some body of text",
        "some body of text:",
        "some initial data",
        "some initial data:",
        "some stuff is set up",
        "some text {prefix}",
        "step0",
        "stuff has been set up",
        "the behave context contains",
        "the behave context contains:",
        "the behave context does not have a parameter \"{param_name}\"",
        "the behave context has a parameter \"{param_name}\"",
        "the default tags \"{default_tags:TagExpression}\"",
        "the directory \"{directory:Path}\" should exist",
        "the directory \"{directory:Path}\" should not exist",
        "the environment variable \"{env_name:Unquoted}\" does not exist",
        "the environment variable \"{env_name:Unquoted}\" exists",
        "the model elements with name and tags",
        "the model elements with name and tags:",
        "the tag \"{tag}\" is not set",
        "the tag \"{tag}\" is set",
        "the tag expression \"{tag_expression:TagExpression}\"",
        "we build an async task",
        "we have behave installed",
        "{amount:Number+} as numbers",
        "{amount:Number} vehicles",
    ],
    "when": [
        "I click on ${environment_variable:w}",
        "I create a symlink from \"{source}\" to \"{dest}\"",
        "I dispatch an async-call with param \"{param}\"",
        "I exercise it work",
        "I inspect the tags of the current scenario",
        "I invite \"{person_name:w}\" for dinner",
        "I provide a/an \"{bigdecimal}\" value as bigdecimal",
        "I provide a/an \"{biginteger}\" value as biginteger",
        "I provide a/an \"{byte}\" value as byte",
        "I provide a/an \"{double}\" value as double",
        "I provide a/an \"{float}\" value as float",
        "I provide a/an \"{int}\" value as int",
        "I provide a/an \"{long}\" value as long",
        "I provide a/an \"{short}\" value as short",
        "I provide a/an \"{word}\" value as word",
        "I provide a/an \"{}\" value as any",
        "I provide a/an {string} value as string",
        "I run \"{command:Unquoted}\"",
        "I run \"{command:Unquoted}\" with encoding=\"{encoding:Unquoted}\"",
        "I run \"{command:Unquoted}\" with locale=\"{locale_value:Unquoted}\"",
        "I run \"{command}\"",
        "I run `{command}`",
        "I run `{command}` with encoding=\"{encoding:Unquoted}\"",
        "I run `{command}` with locale=\"{locale_value:Unquoted}\"",
        "I run the behave model with \"{hint}\"",
        "I select the \"{colors}\" colo(u)r(s)",
        "I select the \"{color}\" theme colo(u)r",
        "I successfully run \"{command:Unquoted}\"",
        "I successfully run `{command}`",
        "I use the environment variable {environment_variable:EnvironmentVar}",
        "^a step passes$",
        "a step passes",
        "an async-step fails",
        "an async-step passes",
        "an async-step raises exception",
        "behave excludes feature files with \"{pattern}\"",
        "behave excludes no feature files",
        "behave includes all feature files",
        "behave includes feature files with \"{pattern}\"",
        "step1",
        "we add some text {suffix}",
        "we can wait for the task in another async step",
        "we implement a test"
    ],
    "then": [
        "I have selected {int} colo(u)r(s)",
        "I wait at most {duration:f} seconds until all async-calls are completed",
        "a dinner reservation for \"{person_name:w}\" and me was made",
        "a file named \"{filename:Path}\" should exist",
        "a file named \"{filename:Path}\" should not exist",
        "an undefined-step snippet should exist for \"{step}\"",
        "an undefined-step snippet should not exist for \"{step}\"",
        "an undefined-step snippets section exists",
        "behave will test it for us!",
        "first step and more",
        "first step is \"{value:Bool}\"",
        "it should fail",
        "it should fail because \"{reason}\"",
        "it should fail with",
        "it should fail with result \"{result:int}\"",
        "it should fail with:",
        "it should pass",
        "it should pass because \"{reason}\"",
        "it should pass with",
        "it should pass with:",
        "it will work",
        "the behave context should contain",
        "the behave context should contain:",
        "the behave hook \"{hook}\" was called",
        "the chardet file encoding for \"{filename}\" should be \"{encoding}\"",
        "the collected result of the async-calls is \"{expected}\"",
        "the command output should be",
        "the command output should be:",
        "the command output should contain",
        "the command output should contain \"{text}\"",
        "the command output should contain \"{text}\" {count:d} times",
        "the command output should contain ANSI escape sequences",
        "the command output should contain exactly",
        "the command output should contain exactly \"{text}\"",
        "the command output should contain exactly:",
        "the command output should contain log records from categories",
        "the command output should contain log records from categories:",
        "the command output should contain the following log record",
        "the command output should contain the following log record:",
        "the command output should contain the following log records",
        "the command output should contain the following log records:",
        "the command output should contain {count:d} times",
        "the command output should contain {count:d} times:",
        "the command output should contain:",
        "the command output should match",
        "the command output should match \"{pattern}\"",
        "the command output should match /{pattern}/",
        "the command output should match:",
        "the command output should not contain",
        "the command output should not contain \"{text}\"",
        "the command output should not contain any ANSI escape sequences",
        "the command output should not contain exactly",
        "the command output should not contain exactly \"{text}\"",
        "the command output should not contain exactly:",
        "the command output should not contain log records from categories",
        "the command output should not contain log records from categories:",
        "the command output should not contain the following log record",
        "the command output should not contain the following log record:",
        "the command output should not contain the following log records",
        "the command output should not contain the following log records:",
        "the command output should not contain:",
        "the command output should not match",
        "the command output should not match \"{pattern}\"",
        "the command output should not match /{pattern}/",
        "the command output should not match:",
        "the command returncode is \"{result:int}\"",
        "the command returncode is non-zero",
        "the command should fail with returncode=\"{result:int}\"",
        "the directory \"{directory:Path}\" should exist",
        "the directory \"{directory:Path}\" should not exist",
        "the environment variable \"{env_name:Unquoted}\" does not exist",
        "the environment variable \"{env_name:Unquoted}\" exists",
        "the file \"{filename:Path}\" should contain",
        "the file \"{filename:Path}\" should contain \"{text}\"",
        "the file \"{filename:Path}\" should not contain",
        "the file \"{filename:Path}\" should not contain \"{text}\"",
        "the file \"{filename:Path}\" should not contain:",
        "the file \"{filename:Path}\" with encoding=\"{encoding:Unquoted}\" should contain \"{text}\"",
        "the file \"{filename:Path}\" with encoding=\"{encoding:Unquoted}\" should contain:",
        "the file \"{filename:Path}\" with encoding=\"{encoding:Unquoted}\" should not contain \"{text}\"",
        "the file \"{filename:Path}\" with encoding=\"{encoding:Unquoted}\" should not contain:",
        "the file \"{filename}\" should contain the log records",
        "the file \"{filename}\" should contain the log records:",
        "the file \"{filename}\" should contain:",
        "the file \"{filename}\" should not contain the log records",
        "the file \"{filename}\" should not contain the log records:",
        "the following active tag combinations are enabled",
        "the following active tag combinations are enabled with inherited tags:",
        "the following active tag combinations are enabled:",
        "the following feature files are selected",
        "the following feature files are selected:",
        "the following files should exist",
        "the following files should not exist",
        "the following scenarios are selected with cmdline",
        "the following scenarios are selected with cmdline:",
        "the number \"{number:d}\" is in the valid range",
        "the numbers \"{number1:d}\" and \"{number2:d}\" are in the valid range",
        "the positive number \"{number:d}\" is in the valid range",
        "the profile colo(u)r should be \"{color}\"",
        "the stored value equals \"{bigdecimal}\" as bigdecimal",
        "the stored value equals \"{biginteger}\" as biginteger",
        "the stored value equals \"{double}\" as double",
        "the stored value equals \"{float}\" as float",
        "the stored value equals \"{int}\" as int",
        "the stored value equals \"{long}\" as long",
        "the stored value equals \"{short}\" as short",
        "the stored value equals \"{word}\" as word",
        "the stored value equals \"{}\" as any",
        "the stored value equals {string} as string",
        "the tag \"{tag}\" is contained",
        "the tag expression selects elements with tags",
        "the tag expression selects elements with tags:",
        "the tag expression selects model elements with",
        "the tag expression selects model elements with:",
        "the text is as expected",
        "the text is substituted as expected",
        "undefined-step snippets should exist for",
        "undefined-step snippets should exist for:",
        "undefined-step snippets should not exist for",
        "undefined-step snippets should not exist for:",
        "we get \"{argument}\" parsed",
        "we should get the {combination}",
        "we will have the expected data",
        "we will have the substituted data"
    ],
    "step": [
        "I am on the profile customisation/settings page",
        "I capture log records",
        "I capture log records with level \"{level}\" or above",
        "I create a log record with",
        "I create a log record with:",
        "I create log records for the following categories",
        "I create log records with",
        "I create log records with:",
        "I define the log record schema",
        "I define the log record schema:",
        "I inspect the following environment variables:",
        "I remove the directory \"{directory:Path}\"",
        "I remove the environment variable \"{env_name:Unquoted}\"",
        "I remove the file \"{filename:Path}\"",
        "I remove the file named \"{filename:Path}\"",
        "I set the context parameter \"{param_name}\" to \"{value}\"",
        "I set the environment variable \"{env_name:Unquoted}\" to \"{env_value:Unquoted}\"",
        "I set the following environment variables:\"",
        "I use \"{log_record_format}\" as log record format",
        "I use the directory \"{directory}\" as working directory",
        "I use the log record configuration",
        "I use the log record configuration:",
        "Russian text",
        "a file named \"{filename:Path}\" does not exist",
        "a file named \"{filename:Path}\" exists",
        "a file named \"{filename}\" exists",
        "a parameter with \"{param:AnyText}\"",
        "an async coroutine step waits \"{duration:f}\" seconds",
        "an async coroutine step waits {duration:f} seconds",
        "an async step is executed",
        "an async-step",
        "an async-step is called",
        "an async-step is called with \"{value:d}",
        "an async-step raises RuntimeError",
        "an async-step should have \"{name}",
        "an async-step waits {duration:f} seconds",
        "an async-step with timeout",
        "an async-step with {name}",
        "an error should fail because \"{reason}\"",
        "an sync-step is called",
        "feature background step_{step_id:d}",
        "note that \"{remark}\"",
        "rule {rule_id:w} background step_{step_id:d}",
        "rule {rule_id:w} scenario_{scenario_id:d} step_{step_id:d}",
        "some async-step is used",
        "some file name \"{filename:Path}\" exists",
        "some parameter with \"{param:Unquoted}\"",
        "the behave context should have a parameter \"{param_name}\"",
        "the behave context should not have a parameter \"{param_name}\"",
        "the directory \"{directory:Path}\" does not exist",
        "the directory \"{directory:Path}\" exists",
        "the file named \"{filename:Path}\" does not exist",
        "the file named \"{filename:Path}\" exists",
        "the following files do not exist",
        "the following files exist",
        "the parameter \"{param_name}\" does not exist in the behave context",
        "the parameter \"{param_name}\" exists in the behave context",
        "unknown categories are ignored in active tags",
        "unknown categories are not ignored in active tags",
        "{name} fails with error",
        "{name} fails with failed",
        "{name} passes",
        "{name} passes with output",
        "{name} passes without output",
        "{name} without output",
        "{word:w} step fails",
        "{word:w} step fails with",
        "{word:w} step fails with \"{message}\"",
        "{word:w} step passes",
        "{word} background step passes",
        "{word} step fails",
        "{word} step passes"
    ]
}

def format_steps_for_prompt(steps: dict) -> str:
    result = []
    for step_type, step_list in steps.items():
        result.append(f"\n{step_type.upper()} steps:")
        for step in step_list:
            result.append(f"  - {step}")
    return "\n".join(result)


## 3. AI Agent

Основная логика агента: формирование промпта и вызов LLM.

In [25]:
SYSTEM_PROMPT = """Ты - эксперт по автоматизации тестирования. Твоя задача - преобразовать ручной тест-кейс в Gherkin формат (.feature файл).

ПРАВИЛА:
1. Используй ТОЛЬКО шаги из предоставленного списка доступных шагов
2. Если нужный шаг отсутствует в списке, используй ближайший подходящий или создай новый шаг в том же стиле
3. Генерируй валидный Gherkin синтаксис
4. Добавляй осмысленные названия Feature и Scenario
5. Используй Given для предусловий, When для действий, Then для проверок
6. Используй And для последовательных шагов одного типа

ФОРМАТ ОТВЕТА:
Верни только .feature файл без дополнительных пояснений.
"""

def create_prompt(test_case: str, available_steps: dict) -> str:
    """Создает промпт для LLM."""
    steps_text = format_steps_for_prompt(available_steps)
    
    return f"""ДОСТУПНЫЕ ШАГИ:
{steps_text}

РУЧНОЙ ТЕСТ-КЕЙС:
{test_case}

Преобразуй этот тест-кейс в Gherkin формат, используя доступные шаги."""


def generate_gherkin(test_case: str, model: str = "gpt-4.1-nano") -> str:
    """Генерирует Gherkin feature файл из текста тест-кейса."""
    
    prompt = create_prompt(test_case, AVAILABLE_STEPS)
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0,  # Низкая температура для более детерминированного вывода
    )
    
    return response.choices[0].message.content


def save_feature(content: str, filename: str = "generated_test.feature"):
    """Сохраняет сгенерированный feature файл."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"✓ Сохранено в {filename}")


print("✓ Agent готов к работе")

✓ Agent готов к работе


## 4. Пример использования

In [26]:
# Пример ручного тест-кейса (обновлен для behave)
test_case = """
Название: Проверка логирования и захвата вывода
1. Создать новую рабочую директорию
2. Создать файл "features/steps/steps.py" с импортом "behave4cmd0.log_steps"
3. Создать feature-файл, который создает лог-запись с категорией "root", уровнем "ERROR" и сообщением "Test Error"
4. Запустить behave для этого feature-файла
5. Проверить, что вывод команды содержит "CAPTURED LOG:"
6. Проверить, что вывод команды содержит "ERROR:root: Test Error"
"""

print("Входной тест-кейс:")
print(test_case)

Входной тест-кейс:

Название: Проверка логирования и захвата вывода
1. Создать новую рабочую директорию
2. Создать файл "features/steps/steps.py" с импортом "behave4cmd0.log_steps"
3. Создать feature-файл, который создает лог-запись с категорией "root", уровнем "ERROR" и сообщением "Test Error"
4. Запустить behave для этого feature-файла
5. Проверить, что вывод команды содержит "CAPTURED LOG:"
6. Проверить, что вывод команды содержит "ERROR:root: Test Error"



In [27]:
# Генерация Gherkin
print("Генерация Gherkin...\n")

gherkin_result = generate_gherkin(test_case)

print("=" * 50)
print("РЕЗУЛЬТАТ:")
print("=" * 50)
print(gherkin_result)

Генерация Gherkin...

РЕЗУЛЬТАТ:
Feature: Проверка логирования и захвата вывода

  Scenario: Создание лог-записи и проверка вывода
    Given I create log records for the following categories
    And I create a log record with:
      | category | root |
      | level    | ERROR |
      | message  | Test Error |
    When I capture log records
    Then the command output should contain "CAPTURED LOG:"
    And the command output should contain "ERROR:root: Test Error"


## 5. Дополнительные примеры

Попробуйте другие тест-кейсы:

In [22]:
# Пример 2: Регистрация пользователя
test_case_registration = """
Название: Регистрация нового пользователя

Предусловия:
- Пользователь не авторизован

Шаги:
1. Открыть главную страницу
2. Нажать кнопку "Регистрация"
3. Заполнить поле "Имя" значением "Иван Петров"
4. Заполнить поле "Email" значением "ivan@test.com"
5. Заполнить поле "Пароль" значением "MyPassword123"
6. Заполнить поле "Подтверждение пароля" значением "MyPassword123"
7. Установить чекбокс "Согласен с условиями"
8. Нажать кнопку "Зарегистрироваться"

Ожидаемый результат:
- Отображается сообщение "Регистрация успешна"
- Пользователь перенаправлен на страницу входа
"""

# Раскомментируйте для генерации:
# print(generate_gherkin(test_case_registration))

Feature: Регистрация нового пользователя

  Scenario: Успешная регистрация нового пользователя
    Given Пользователь не авторизован
    When я открываю главную страницу
    And я нажимаю кнопку "Регистрация"
    And я заполняю поле "Имя" значением "Иван Петров"
    And я заполняю поле "Email" значением "ivan@test.com"
    And я заполняю поле "Пароль" значением "MyPassword123"
    And я заполняю поле "Подтверждение пароля" значением "MyPassword123"
    And я устанавливаю чекбокс "Согласен с условиями"
    And я нажимаю кнопку "Зарегистрироваться"
    Then отображается сообщение "Регистрация успешна"
    And пользователь перенаправлен на страницу входа


In [23]:
# Пример 3: Негативный сценарий - неверный пароль
test_case_negative = """
Название: Ошибка при неверном пароле

Шаги:
1. Открыть страницу логина
2. Ввести корректный email user@test.com
3. Ввести неверный пароль wrongpassword
4. Нажать кнопку Войти
5. Проверить что отображается ошибка "Неверный email или пароль"
6. Проверить что пользователь остался на странице логина
"""

# Раскомментируйте для генерации:
print(generate_gherkin(test_case_negative))

Feature: Login error handling with incorrect password

Scenario: Display error message when user enters wrong password
  Given I am testing stuff
  When I open the login page
  And I provide a/an "user@test.com" value as string
  And I provide a/an "wrongpassword" value as string
  And I click on the "Войти" button
  Then I should see the error message "Неверный email или пароль"
  And I should remain on the login page
